# 04. Spark SQL

- Spark SQL basics & Delta Lake (1 hour)
  - Temporary view
  - SQL Queries
  - Spark SQL vs traditional SQL
  - Basics of delta lake


## Create Dataframe

In [0]:
# %python
from pyspark.sql import Row
from pyspark.sql import functions as F
from datetime import datetime

sales_data = [
    Row(order_id=1, customer_id=101, product="Laptop", category="Electronics", amount=75000, order_date="2026-01-10"),
    Row(order_id=2, customer_id=102, product="Mobile", category="Electronics", amount=30000, order_date="2026-01-12"),
    Row(order_id=3, customer_id=101, product="Mouse", category="Accessories", amount=1200, order_date="2026-01-15"),
    Row(order_id=4, customer_id=103, product="Chair", category="Furniture", amount=8500, order_date="2026-02-01"),
    Row(order_id=5, customer_id=104, product="Desk", category="Furniture", amount=15000, order_date="2026-02-10"),
    Row(order_id=6, customer_id=102, product="Keyboard", category="Accessories", amount=2500, order_date="2026-02-15")
]

customer_data = [
    Row(customer_id=101, customer_name="Alice", city="Bangalore"),
    Row(customer_id=102, customer_name="Bob", city="Chennai"),
    Row(customer_id=103, customer_name="Cathy", city="Mumbai"),
    Row(customer_id=104, customer_name="Diana", city="Pune"),
    Row(customer_id=105, customer_name="Evan", city="Hyderabad")
]

sales_df = spark.createDataFrame(sales_data) \
    .withColumn("order_date", F.to_date("order_date"))

customers_df = spark.createDataFrame(customer_data)

display(sales_df)
display(customers_df)

## Temp View

In [0]:
# %python
sales_df.createOrReplaceTempView("sales")
customers_df.createOrReplaceTempView("customers")

In [0]:
%sql
-- %sql
SELECT *
FROM sales;

In [0]:
%sql
-- %sql
SELECT *
FROM customers;

## SQL Operations

Select columns

In [0]:
%sql
SELECT
    order_id,
    customer_id,
    product,
    amount
FROM sales;

Filter Columns

In [0]:
%sql
-- %sql
SELECT
    order_id,
    product,
    category,
    amount
FROM sales
WHERE amount > 10000;

Sort

In [0]:
%sql
-- %sql
SELECT
    order_id,
    product,
    amount
FROM sales
ORDER BY amount DESC;

Aggregations

In [0]:
%sql
-- %sql
SELECT
    category,
    COUNT(*) AS total_orders,
    SUM(amount) AS total_sales,
    AVG(amount) AS average_order_value,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount
FROM sales
GROUP BY category
ORDER BY total_sales DESC;

Joins

In [0]:
%sql
-- %sql
SELECT
    s.order_id,
    c.customer_name,
    c.city,
    s.product,
    s.category,
    s.amount,
    s.order_date
FROM sales s
INNER JOIN customers c
    ON s.customer_id = c.customer_id;

In [0]:
%sql
-- %sql
SELECT
    c.customer_id,
    c.customer_name,
    c.city,
    s.order_id,
    s.product,
    s.amount
FROM customers c
LEFT JOIN sales s
    ON c.customer_id = s.customer_id
ORDER BY c.customer_id;

Apply function

In [0]:
%sql
-- %sql
SELECT
    order_id,
    product,
    order_date,
    YEAR(order_date) AS order_year,
    MONTH(order_date) AS order_month,
    DAY(order_date) AS order_day,
    DATE_ADD(order_date, 7) AS delivery_estimate_date
FROM sales;

In [0]:
%sql
-- %sql
SELECT
    order_id,
    customer_id,
    product,
    category,
    amount,
    RANK() OVER (
        PARTITION BY category
        ORDER BY amount DESC
    ) AS category_rank
FROM sales;

## Spark SQL vs Traditional SQL Difference

> Spark SQL allows us to use SQL syntax on distributed data.
> The query looks like traditional SQL, but Spark converts it into a distributed execution plan and runs it across multiple executors.
> This makes Spark SQL suitable for large-scale data processing.

|Area|	Spark SQL|	Traditional SQL|
|--|--|--|
|Execution|	Distributed across cluster|	Usually single database engine or MPP database|
|Data size|	Handles very large-scale data|	Depends on database capacity|
|Storage|	Can query files, tables, data lakes|	Usually queries database tables|
|Optimization|	Catalyst optimizer|	Database-specific optimizer|
|Processing style|	Batch, streaming, lakehouse workloads|	OLTP or analytical workloads|
|Data formats|	Parquet, Delta, ORC, JSON, CSV|	Internal database storage|
|Schema|	Can work with schema-on-read|	Usually schema-on-write|
|Updates|	Best with Delta Lake, Iceberg, Hudi|	Native support in databases|
|Use case|	Big data ETL, analytics, ML pipelines|	Application transactions, reporting, BI|

## Delta Lake Basics

In [0]:
# %python
sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/pyspark_training/raw_data/delta_tables/")

In [0]:
# %python
delta_sales_df = spark.read \
    .format("delta") \
    .load("/Volumes/workspace/pyspark_training/raw_data/delta_tables/")

display(delta_sales_df)

In [0]:
%sql
-- %sql
DROP TABLE IF EXISTS  workspace.default.demo_delta_sales;

In [0]:
%sql
CREATE TABLE  workspace.default.demo_delta_sales
(
    order_id INT,
    customer_id INT,
    product STRING,
    category STRING
)



In [0]:
%sql
-- %sql
SELECT *
FROM workspace.default.demo_delta_sales;

In [0]:
%sql
INSERT INTO workspace.default.demo_delta_sales
SELECT 1,1,'product1','category1' UNION ALL
SELECT 2,2,'product2','category2' UNION ALL
SELECT 3,3,'product3','category3' UNION ALL
SELECT 4,4,'product4','category4'
    

In [0]:
%sql
UPDATE workspace.default.demo_delta_sales
SET category = 'category2_new'
WHERE
    order_id = 2

In [0]:
%sql
DELETE
FROM workspace.default.demo_delta_sales
WHERE order_id = 3

In [0]:
%sql
-- %sql
DESCRIBE DETAIL demo_delta_sales;

In [0]:
%sql
-- %sql
DESCRIBE HISTORY demo_delta_sales;